# 02 · Validación y carga a Supabase

Este notebook documenta, paso a paso, lo que **realmente** hace `cargar_csv_supabase.py` (ETL) y `cargar_coordenadas.py` (geocodificación), para que quede como evidencia reproducible y trazable ante el jurado.

**Nota de transparencia:** no hay un notebook de limpieza previa separado — la limpieza ocurre *dentro* del script de carga (columnas obligatorias, código DANE válido, normalización de texto), no como un paso independiente sobre el CSV crudo.

## 1. Lectura tolerante a codificación y validación de columnas obligatorias

In [ ]:
import pandas as pd

RUTA_CSV = "../Reporte_Hurto_por_Modalidades_Policía_Nacional_20260711.csv"

try:
    df = pd.read_csv(RUTA_CSV, encoding="utf-8", low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(RUTA_CSV, encoding="latin-1", low_memory=False)

df.columns = df.columns.str.strip()
print(f"Columnas: {list(df.columns)}")

columnas_obligatorias = ["CODIGO DANE", "FECHA HECHO", "TIPO DE HURTO", "CANTIDAD"]
faltantes = [c for c in columnas_obligatorias if c not in df.columns]
assert not faltantes, f"Faltan columnas obligatorias: {faltantes}"

## 2. Limpieza del código DANE (control real de completitud/validez)

In [ ]:
total_inicial = len(df)
df["CODIGO DANE"] = pd.to_numeric(df["CODIGO DANE"], errors="coerce").fillna(0).astype(int)
df = df[df["CODIGO DANE"] > 0]
print(f"Filas válidas: {len(df):,} de {total_inicial:,} originales "
      f"({total_inicial - len(df)} descartadas por código DANE inválido/ausente)")

## 3. Normalización de texto (municipio/departamento)

In [ ]:
df["MUNICIPIO_NORM"] = df["MUNICIPIO"].astype(str).str.replace(" (CT)", "", regex=False).str.strip().str.upper()
df["DEPARTAMENTO_NORM"] = df["DEPARTAMENTO"].astype(str).str.strip().str.upper()
df[["MUNICIPIO", "MUNICIPIO_NORM", "DEPARTAMENTO", "DEPARTAMENTO_NORM"]].drop_duplicates().head(10)

## 4. Clasificación de tipo de hurto (heurística por palabra clave)

**Regla real del ETL:** si el texto de `TIPO DE HURTO` contiene la palabra "RESIDENCIA" → `ART239-RES`; en cualquier otro caso → `ART239-COM`. Esta celda cuantifica cuántas filas caen en cada categoría, para dimensionar el sesgo potencial que esto introduce (ver `docs/conclusiones.md`).

In [ ]:
def clasificar_tipo(texto):
    texto = str(texto).upper()
    return "ART239-RES" if "RESIDENCIA" in texto else "ART239-COM"

df["tipo_hurto_codigo"] = df["TIPO DE HURTO"].apply(clasificar_tipo)
df["tipo_hurto_codigo"].value_counts()

In [ ]:
# Valores originales que caen en "comercial" por defecto (para revisar si es correcto)
df.loc[df["tipo_hurto_codigo"] == "ART239-COM", "TIPO DE HURTO"].value_counts()

## 5. Armas/medios: catálogo dinámico y valor por defecto

In [ ]:
df["ARMAS MEDIOS"] = df.get("ARMAS MEDIOS", pd.Series(dtype=str)).fillna("NO REPORTADO").astype(str).str.strip().str.upper()
df["ARMAS MEDIOS"].replace("", "NO REPORTADO", inplace=True)
df["ARMAS MEDIOS"].value_counts().head(10)

## 6. Fechas: parseo día/mes/año y filas con fecha no interpretable

**Riesgo real documentado en `conclusiones.md`:** en el script de producción, si una fecha no se puede parsear, se reemplaza silenciosamente por la fecha del sistema. Esta celda solo **cuantifica** cuántas filas caerían en ese caso, sin aplicar el reemplazo, para dimensionar el riesgo.

In [ ]:
fechas_parseadas = pd.to_datetime(df["FECHA HECHO"].astype(str).str[:10], dayfirst=True, errors="coerce")
no_parseables = fechas_parseadas.isna().sum()
print(f"Fechas no interpretables: {no_parseables} de {len(df)} ({no_parseables/len(df)*100:.2f}%)")

## 7. Filas potencialmente duplicadas (no se eliminan en producción, solo se cuantifican aquí)

In [ ]:
cols_clave = ["CODIGO DANE", "FECHA HECHO", "TIPO DE HURTO", "CANTIDAD"]
n_duplicados = df.duplicated(subset=cols_clave).sum()
print(f"Filas duplicadas detectadas (no eliminadas por el ETL actual): {n_duplicados}")

## 8. Enriquecimiento geográfico (`cargar_coordenadas.py`)

Paso posterior e independiente del ETL principal: geocodifica los municipios de `ubicaciones` que aún no tienen `lat`/`lng`, usando **Nominatim (OpenStreetMap)** vía `geopy`, respetando el límite de 1 solicitud/segundo del servicio.

```python
from geopy.geocoders import Nominatim
geo = Nominatim(user_agent="seph_colombia")
loc = geo.geocode(f"{municipio}, {departamento}, Colombia", timeout=10)
# loc.latitude, loc.longitude → se guardan en ubicaciones.lat / ubicaciones.lng
```

No se ejecuta aquí para no golpear el servicio de geocodificación en cada corrida del notebook; se documenta como referencia de lo que ya corre en `cargar_coordenadas.py`.

## 9. Resumen para `docs/conclusiones.md`

> Completa con los resultados reales de las celdas anteriores:
> - Filas descartadas por código DANE inválido: `___`
> - % clasificado como comercial por defecto (revisar si es razonable): `___`
> - Fechas no interpretables detectadas: `___`
> - Filas potencialmente duplicadas: `___`